# Congressional civility & "toxicity" — interactive explorer

Explore the *decline of comity between the parties* in the U.S. Congressional Record,
**1873–2026**, and interact with the underlying data and plots.

**Data**
- 1873–2017: Stanford *hein* corpus (Gentzkow–Shapiro–Taddy), speaker-segmented with party.
- 2017–present: GovInfo CREC (whole-day package zips), segmented into speaker turns; party
  from MODS `congMember@party`.

**What we measure** (all rates are per 1,000 words, split by speaker party):
- *Comity / deference* — courtesy phrases ("the gentleman from", "I thank the gentleman").
- *Hostility / attack* — attack/insult lexicon.
- *Profanity* — a comprehensive tiered lexicon (mild / strong / slurs).
- *Cross-party reference tone* — out-group references + **directed** hostility/comity near them.
- *Sentiment / "toxicity"* — VADER sentence-level compound + negative-affect share (see the
  toxicity section below for exactly what this means and how we validate it).

**Keyword matching is fuzzy.** Lexicon terms are matched with morphological variants (plurals
and verb forms: "colleague"→"colleagues", "corrupt"→"corrupting"; irregular plurals like
"gentleman"→"gentlemen"), so recall doesn't hinge on listing every inflection. (OCR noise in the
pre-2011 scanned text is a known residual limitation — see caveats.)

> **On the word "toxicity."** There is no single ground-truth "toxicity" label for a
> 150-year speech corpus. Here *toxicity* is operationalised **lexically**: hostility + profanity
> rates and VADER negative-affect share. That is fast, transparent, and reproducible over ~13M
> turns — but lexical measures miss sarcasm, quotation, and context. The final section validates
> the lexical proxy against a real transformer toxicity model (Detoxify) on a sample.

In [ ]:
import sys, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Make the analysis package importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from analysis.plotting import theme, charts   # shared Substack-style toolkit
theme.apply()

METRICS = ROOT / "data" / "processed" / "metrics" / "civility_metrics.parquet"
assert METRICS.exists(), f"Run `python -m analysis.run aggregate` first; missing {METRICS}"
m = pd.read_parquet(METRICS)
print(f"metrics rows: {len(m)}  | congresses {m.congress.min()}–{m.congress.max()}"
      f" (years {m.year.min()}–{m.year.max()})")
m.head()

## 1. Re-aggregate to (year, party)

The metrics table is stored at `(congress, chamber, party)` granularity with **raw hit
counts + word totals**, so we can re-aggregate to any grouping while keeping rates
word-weighted (never averaging rates-of-rates).

In [ ]:
HIT_COLS = ["comity_hits","hostility_hits","profanity_hits","profanity_slurs_hits",
            "outgroup_refs","democrat_party_pej","directed_comity_hits",
            "directed_hostility_hits","words"]

def by_year_party(df):
    g = df.groupby(["year","party"], as_index=False)[HIT_COLS].sum()
    w = g["words"].replace(0, 1)
    g["comity_per_1k"]            = 1000*g["comity_hits"]/w
    g["hostility_per_1k"]         = 1000*g["hostility_hits"]/w
    g["profanity_per_1k"]         = 1000*g["profanity_hits"]/w
    g["slurs_per_1k"]             = 1000*g["profanity_slurs_hits"]/w
    g["outgroup_ref_per_1k"]      = 1000*g["outgroup_refs"]/w
    g["democrat_party_pej_per_1k"]= 1000*g["democrat_party_pej"]/w
    g["directed_hostility_per_1k"]= 1000*g["directed_hostility_hits"]/w
    return g

g = by_year_party(m)
g[g.party.isin(["D","R"])].tail(6)

## 2. By party **and** chamber (House vs Senate)

The same metrics split four ways — party (colour) × chamber (House solid, Senate dashed).
Aggregated from raw hits/words at `(congress, chamber, party)` so rates stay word-weighted.

In [ ]:
CHAM_STYLE = {"house": "-", "senate": "--"}
CHAM_LABEL = {"house": "House", "senate": "Senate"}

def by_year_chamber_party(df):
    g = df.groupby(["year","chamber","party"], as_index=False)[HIT_COLS].sum()
    w = g["words"].replace(0, 1)
    for c in ["comity","hostility","profanity"]:
        g[f"{c}_per_1k"] = 1000*g[f"{c}_hits"]/w
    g["directed_hostility_per_1k"] = 1000*g["directed_hostility_hits"]/w
    g["outgroup_ref_per_1k"] = 1000*g["outgroup_refs"]/w
    return g

gc = by_year_chamber_party(m)

def plot_by_chamber(metric, title, ylabel="hits per 1,000 words"):
    fig, ax = charts.new_figure(figsize=(11, 6))
    for p in ("D","R"):
        for ch in ("house","senate"):
            sub = gc[(gc.party==p) & (gc.chamber==ch)].sort_values("year")
            if sub.empty: continue
            charts.line(ax, sub["year"], sub[metric], color=theme.PARTY_COLORS[p],
                        label=f"{theme.PARTY_LABELS[p]} — {CHAM_LABEL[ch]}",
                        linestyle=CHAM_STYLE[ch], linewidth=2.0, markersize=3)
    charts.marker_line(ax, 2017)
    charts.style_axes(ax, f"{title} — by party & chamber", "Year", ylabel,
                      subtitle="solid = House, dashed = Senate")
    ax.legend(loc="best", frameon=False, labelcolor=theme.TEXT, fontsize=9)
    plt.show()

plot_by_chamber("hostility_per_1k", "Hostility / attack language")
plot_by_chamber("comity_per_1k", "Comity / deference phrases")

## 3. Plot any metric by party (overall)

Change `METRIC` to any `*_per_1k` column and re-run.

In [ ]:
def plot_metric(metric, title, ylabel="hits per 1,000 words", parties=("D","R")):
    fig, ax = charts.new_figure(figsize=(10, 5.5))
    for p in parties:
        sub = g[g.party == p].sort_values("year")
        charts.line(ax, sub["year"], sub[metric], color=theme.PARTY_COLORS[p],
                    label=theme.PARTY_LABELS[p])
    charts.marker_line(ax, 2017)  # hein -> GovInfo source boundary
    charts.style_axes(ax, title, "Year", ylabel)
    ax.legend(loc="best", frameon=False, labelcolor=theme.TEXT)
    plt.show()
    return fig

METRIC = "comity_per_1k"
plot_metric(METRIC, "Comity / deference phrases");

In [ ]:
# The six headline panels at a glance.
for metric, title in [
    ("comity_per_1k", "Comity / deference phrases"),
    ("hostility_per_1k", "Hostility / attack language"),
    ("directed_hostility_per_1k", "Hostility directed at the other party"),
    ("outgroup_ref_per_1k", "References to the other party"),
    ("profanity_per_1k", "Profanity"),
    ("democrat_party_pej_per_1k", '"Democrat party" pejorative'),
]:
    plot_metric(metric, title);

## 4. Toxicity — are we doing it right?

"Toxicity" here = the **negative-civility** signals: hostility + profanity lexicon rates and
**VADER** negative sentiment. Two methodological points we get right:

1. **Sentence-level VADER.** VADER's `compound` score saturates on long text, so scoring a whole
   speech (or a fixed truncation of it) is biased. `Scorers._sentiment` splits each turn into
   sentences, scores each, and averages — the granularity VADER is designed for — and reports the
   mean **negative-affect share**.
2. **Word-weighted rates.** Lexicon counts are normalised per 1,000 words, never compared as raw
   counts across eras of differing verbosity.

Below we score a live sample of raw turns and check the lexical signals and VADER agree.

In [ ]:
from analysis.score.scorers import Scorers

TURN_FILES = sorted(glob.glob(str(ROOT / "data" / "interim" / "turns" / "*.parquet")))
assert TURN_FILES, "No turn parquet files; run the ingest step first."

# Sample recent turns (bulk GovInfo if present, else the newest available file).
recent = [f for f in TURN_FILES if "govinfo_bulk_" in f] or TURN_FILES[-1:]
sample = (pd.read_parquet(recent[-1], columns=["party","text","is_procedural"])
            .query("~is_procedural and text.str.len() > 0", engine="python")
            .sample(2000, random_state=0))

scorer = Scorers(use_sentiment=True)   # VADER on (this smaller sample only)
scored = sample.assign(**pd.DataFrame(
    [scorer.score_turn(t, p) for t, p in zip(sample.text, sample.party)],
    index=sample.index)[["n_words","hostility_hits","profanity_hits","sentiment","neg_share"]])
scored["hostility_per_1k"] = 1000*scored.hostility_hits/scored.n_words.clip(lower=1)
scored["profanity_per_1k"] = 1000*scored.profanity_hits/scored.n_words.clip(lower=1)
scored[["party","n_words","hostility_per_1k","profanity_per_1k","sentiment","neg_share"]].describe()

In [ ]:
# Do the independent negativity signals agree? (lexical hostility/profanity vs VADER)
corr = scored[["hostility_per_1k","profanity_per_1k","neg_share","sentiment"]].corr(method="spearman")
print("Spearman correlations among negativity signals:")
try:
    display(corr.round(2))          # rich table in Jupyter
except NameError:
    print(corr.round(2))            # plain output elsewhere
print("\nExpect: hostility & neg_share positively correlated; sentiment (compound) negatively "
      "correlated with both. Agreement between independent methods supports the lexical proxy.")

## 5. Optional: validate against a real toxicity model (Detoxify)

The cell below is **optional and heavy** (installs `torch` + `transformers` and downloads a
model). It runs the Detoxify `original` classifier on a small sample and correlates its
`toxicity` probability with our lexical proxy. A positive correlation is evidence the fast
lexical measure tracks a bona-fide toxicity model — i.e. we're measuring the right thing.

Uncomment and run only if you want the validation (it does not affect the pipeline).

In [ ]:
# --- OPTIONAL: real transformer toxicity validation (slow; needs internet + torch) ---
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "detoxify"], check=True)
# from detoxify import Detoxify
# val = scored.sample(200, random_state=1).copy()
# tox = Detoxify("original").predict(list(val.text.str.slice(0, 2000)))["toxicity"]
# val["detoxify_toxicity"] = tox
# print("Spearman(lexical hostility/1k, Detoxify toxicity):",
#       round(val["hostility_per_1k"].corr(val["detoxify_toxicity"], method="spearman"), 3))
# print("Spearman(VADER neg_share,     Detoxify toxicity):",
#       round(val["neg_share"].corr(val["detoxify_toxicity"], method="spearman"), 3))
print("Optional Detoxify validation cell — uncomment the lines above to run it.")

## 6. Caveats

- **Lexicons are literal.** They miss sarcasm, negation ("not corrupt"), and **quotation** — a
  member reading an opponent's words is scored as if they were their own. Profanity/attacks on
  the floor also often appear inside quoted material.
- **VADER is lexical**, tuned for modern English; 19th-century phrasing is out-of-distribution.
- **Source discontinuity at 2017** (hein → GovInfo) — the dotted line on every chart. Treat
  cross-boundary jumps cautiously; compare within-source trends.
- **Segmentation error** on the GovInfo side (regex speaker splitting) is measured but non-zero.
- **Party = speaker's party**, not the target's; "directed" measures use a text window around
  out-group references as a heuristic for who is being addressed.